# Pipeline d'Analyse de Trajectoire de Drone

Ce notebook consolide l'ensemble du projet "Projet AI et TI" en un seul document interactif.
Il permet d'analyser le comportement des drones (Normal, Suspect, Hostile) à partir de données de tracking (CSV) et d'une vidéo.

## Fonctionnalités
1. **Lissage de Trajectoire**: Utilise un Filtre de Kalman pour réduire le bruit.
2. **Extraction de Caractéristiques**: Calcule la vitesse, l'accélération et les virages.
3. **Prédiction de Trajectoire**: Utilise la régression polynomiale pour prédire les mouvements futurs.
4. **Analyse de Comportement**: Système expert detectant les comportements hostiles.
5. **Visualisation**: Génère une vidéo annotée avec des alertes visuelles et sonores.

## 1. Importation des Librairies

In [10]:
import cv2
import json
import csv
import numpy as np
import wave
import struct
import subprocess
import os
import winsound
import threading
from collections import defaultdict
from collections import deque
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuration pour l'affichage dans le notebook
%matplotlib inline

## 2. Module: Filtre de Kalman

In [11]:
class DroneKalmanFilter:
    """
    Implémente un filtre de Kalman pour le suivi fluide de drone 2D.
    Modèle à vitesse constante (Position X, Y, Vitesse Vx, Vy).
    """
    
    def __init__(self, dt=1/30.0, u_x=1, u_y=1, std_acc=1, x_std_meas=0.1, y_std_meas=0.1):
        """
        Initialise le filtre de Kalman.
        
        Args:
            dt (float): pas de temps (delta t)
            u_x (float): accélération en x
            u_y (float): accélération en y
            std_acc (float): bruit de processus (accélération)
            x_std_meas (float): bruit de mesure en x
            y_std_meas (float): bruit de mesure en y
        """
        self.dt = dt
        self.u = np.matrix([[u_x],[u_y]])
        self.x = np.matrix([[0], [0], [0], [0]])  # État initial: x, y, vx, vy
        
        # Matrice de transition A
        self.A = np.matrix([[1, 0, self.dt, 0],
                            [0, 1, 0, self.dt],
                            [0, 0, 1, 0],
                            [0, 0, 0, 1]])
        
        # Matrice de commande B
        self.B = np.matrix([[0.5 * self.dt**2, 0],
                            [0, 0.5 * self.dt**2],
                            [self.dt, 0],
                            [0, self.dt]])
        
        # Matrice d'observation H
        self.H = np.matrix([[1, 0, 0, 0],
                            [0, 1, 0, 0]])
        
        # Covariance du bruit de processus Q
        self.Q = np.matrix([[(self.dt**4)/4, 0, (self.dt**3)/2, 0],
                            [0, (self.dt**4)/4, 0, (self.dt**3)/2],
                            [(self.dt**3)/2, 0, self.dt**2, 0],
                            [0, (self.dt**3)/2, 0, self.dt**2]]) * std_acc**2
        
        # Covariance du bruit de mesure R
        self.R = np.matrix([[x_std_meas**2, 0],
                            [0, y_std_meas**2]])
        
        # Matrice de covariance de l'erreur P
        self.P = np.eye(self.A.shape[1])
        
        self.initialized = False
        
    def predict(self):
        """Étape de prédiction"""
        # Prédiction de l'état
        self.x = self.A * self.x + self.B * self.u
        # Prédiction de la covariance de l'erreur
        self.P = self.A * self.P * self.A.T + self.Q
        return self.x[0,0], self.x[1,0]

    def update(self, z_x, z_y):
        """Étape de mise à jour avec nouvelle mesure"""
        if not self.initialized:
            # Initialisation avec la première mesure
            self.x[0,0] = z_x
            self.x[1,0] = z_y
            self.initialized = True
            return z_x, z_y
            
        # Prédiction d'abord
        self.predict()
        
        # Mesure
        z = np.matrix([[z_x], [z_y]])
        
        # Calcul de S = H*P*H' + R
        S = self.H * self.P * self.H.T + self.R
        
        # Calcul du gain de Kalman K = P*H'*S^-1
        K = self.P * self.H.T * np.linalg.inv(S)
        
        # Mise à jour de l'état x = x + K*(z - H*x)
        y = z - self.H * self.x  # Innovation résiduelle
        self.x = self.x + K * y
        
        # Mise à jour de la covariance P = (I - K*H)*P
        I = np.eye(self.H.shape[1])
        self.P = (I - K * self.H) * self.P
        
        return self.x[0,0], self.x[1,0]
        
    def predict_future(self, n_steps=10):
        """Prédit les n prochaines positions sans mise à jour"""
        predictions = []
        x_temp = self.x.copy()
        P_temp = self.P.copy()
        
        for _ in range(n_steps):
            x_temp = self.A * x_temp + self.B * self.u
            P_temp = self.A * P_temp * self.A.T + self.Q
            predictions.append((x_temp[0,0], x_temp[1,0]))
            
        return predictions

## 3. Module: Extraction de Caractéristiques (Mouvement)

In [12]:
class MotionFeatureExtractor:
    """
    Extrait les caractéristiques de mouvement (vitesse, accélération, direction)
    à partir des positions historiques.
    """
    
    def __init__(self, window_size=5):
        self.window_size = window_size
        self.positions = deque(maxlen=window_size)
        self.timestamps = deque(maxlen=window_size)
    
    def add_position(self, x, y, timestamp):
        self.positions.append((x, y))
        self.timestamps.append(timestamp)
    
    def get_velocity(self):
        """Calcule la vitesse instantanée (pixels/frame)"""
        if len(self.positions) < 2:
            return {'vx': 0, 'vy': 0, 'magnitude': 0, 'direction': 0}
        
        p1 = self.positions[-2]
        p2 = self.positions[-1]
        t1 = self.timestamps[-2]
        t2 = self.timestamps[-1]
        
        dt = t2 - t1
        if dt == 0: return {'vx': 0, 'vy': 0, 'magnitude': 0, 'direction': 0}
        
        vx = (p2[0] - p1[0]) / dt
        vy = (p2[1] - p1[1]) / dt
        
        magnitude = np.sqrt(vx**2 + vy**2)
        direction = np.degrees(np.arctan2(vy, vx))
        
        return {'vx': vx, 'vy': vy, 'magnitude': magnitude, 'direction': direction}
    
    def get_average_velocity(self):
        """Calcule la vitesse moyenne sur la fenêtre"""
        if len(self.positions) < 2:
            return {'magnitude': 0}
            
        p_start = self.positions[0]
        p_end = self.positions[-1]
        t_start = self.timestamps[0]
        t_end = self.timestamps[-1]
        
        dt = t_end - t_start
        if dt == 0: return {'magnitude': 0}
        
        dist = np.sqrt((p_end[0] - p_start[0])**2 + (p_end[1] - p_start[1])**2)
        return {'magnitude': dist / dt}
        
    def get_acceleration(self):
        """Calcule l'accélération (changement de vitesse)"""
        if len(self.positions) < 3:
            return {'ax': 0, 'ay': 0, 'magnitude': 0}
            
        # Vitesse au temps t-1
        dt1 = self.timestamps[-2] - self.timestamps[-3]
        vx1 = (self.positions[-2][0] - self.positions[-3][0]) / dt1
        vy1 = (self.positions[-2][1] - self.positions[-3][1]) / dt1
        
        # Vitesse au temps t
        dt2 = self.timestamps[-1] - self.timestamps[-2]
        vx2 = (self.positions[-1][0] - self.positions[-2][0]) / dt2
        vy2 = (self.positions[-1][1] - self.positions[-2][1]) / dt2
        
        # Accélération
        dt = (dt1 + dt2) / 2
        ax = (vx2 - vx1) / dt
        ay = (vy2 - vy1) / dt
        magnitude = np.sqrt(ax**2 + ay**2)
        
        return {'ax': ax, 'ay': ay, 'magnitude': magnitude}
    
    def get_direction_change(self):
        """Calcule le changement de direction (en degrés)"""
        if len(self.positions) < 3:
            return 0
            
        # Vecteur 1
        v1x = self.positions[-2][0] - self.positions[-3][0]
        v1y = self.positions[-2][1] - self.positions[-3][1]
        
        # Vecteur 2
        v2x = self.positions[-1][0] - self.positions[-2][0]
        v2y = self.positions[-1][1] - self.positions[-2][1]
        
        # Angle entre les vecteurs
        angle1 = np.degrees(np.arctan2(v1y, v1x))
        angle2 = np.degrees(np.arctan2(v2y, v2x))
        
        diff = abs(angle2 - angle1)
        if diff > 180:
            diff = 360 - diff
            
        return diff

    def get_all_features(self):
        return {
            'instant_velocity': self.get_velocity(),
            'average_velocity': self.get_average_velocity(),
            'acceleration': self.get_acceleration(),
            'direction_change': self.get_direction_change()
        }

## 4. Module: Prédiction de Trajectoire (ML)

In [13]:
class TrajectoryPredictor:
    """
    Prédit la trajectoire future d'un drone en utilisant la régression polynomiale.
    """
    
    def __init__(self, history_size=15, degree=2):
        self.history_size = history_size
        self.degree = degree
        
        self.x_positions = deque(maxlen=history_size)
        self.y_positions = deque(maxlen=history_size)
        self.frames = deque(maxlen=history_size)
        
        self.model_x = LinearRegression()
        self.model_y = LinearRegression()
        self.poly_features = PolynomialFeatures(degree=degree)
        
        self.is_trained = False
    
    def add_position(self, x, y, frame):
        self.x_positions.append(x)
        self.y_positions.append(y)
        self.frames.append(frame)
        
        # Entraîner si on a assez de données
        if len(self.frames) >= min(5, self.history_size):
            self._train()
    
    def _train(self):
        if len(self.frames) < 3:
            return
        
        # Préparer les données
        frames_array = np.array(self.frames).reshape(-1, 1)
        x_array = np.array(self.x_positions)
        y_array = np.array(self.y_positions)
        
        # Transformation polynomiale
        frames_poly = self.poly_features.fit_transform(frames_array)
        
        # Entraînement
        self.model_x.fit(frames_poly, x_array)
        self.model_y.fit(frames_poly, y_array)
        
        self.is_trained = True
    
    def predict(self, n_steps=10):
        if not self.is_trained or len(self.frames) == 0:
            return []
        
        # Frames futures
        last_frame = self.frames[-1]
        future_frames = np.array([last_frame + i + 1 for i in range(n_steps)]).reshape(-1, 1)
        
        # Transformation polynomiale
        future_frames_poly = self.poly_features.transform(future_frames)
        
        # Prédiction
        x_pred = self.model_x.predict(future_frames_poly)
        y_pred = self.model_y.predict(future_frames_poly)
        
        predictions = [(x, y) for x, y in zip(x_pred, y_pred)]
        
        return predictions
    
    def get_trajectory_curvature(self):
        if len(self.x_positions) < 3:
            return 0
        
        # Utiliser les 3 derniers points
        positions = list(zip(self.x_positions, self.y_positions))[-3:]
        
        # Calculer les vecteurs
        v1 = np.array([positions[1][0] - positions[0][0], 
                       positions[1][1] - positions[0][1]])
        v2 = np.array([positions[2][0] - positions[1][0], 
                       positions[2][1] - positions[1][1]])
        
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        
        if norm1 < 1e-6 or norm2 < 1e-6:
            return 0
        
        cos_angle = np.dot(v1, v2) / (norm1 * norm2)
        cos_angle = np.clip(cos_angle, -1, 1)
        angle = np.arccos(cos_angle)
        
        return angle

## 5. Module: Analyse Comportementale

In [14]:
class BehaviorAnalyzer:
    """
    Analyse le comportement d'un drone et détecte les patterns hostiles.
    """
    
    # Seuils de détection
    THRESHOLDS = {
        'high_speed': 50,           # pixels/frame
        'high_acceleration': 15,    
        'sharp_turn': 45,            # degrés
        'erratic_behavior_count': 3, 
        'approach_distance': 100,    
        'loitering_frames': 30       
    }
    
    STATUS_NORMAL = "NORMAL"
    STATUS_SUSPECT = "SUSPECT"
    STATUS_HOSTILE = "HOSTILE"
    
    def __init__(self):
        self.suspicious_events = []
        self.behavior_history = []
    
    def analyze(self, motion_features, trajectory_info=None):
        reasons = []
        threat_score = 0
        
        # Récupération des features
        instant_vel = motion_features.get('instant_velocity', {})
        accel = motion_features.get('acceleration', {})
        dir_change = motion_features.get('direction_change', 0)
        
        # 1. Vitesse excessive
        speed = instant_vel.get('magnitude', 0)
        if speed > self.THRESHOLDS['high_speed']:
            reasons.append(f"Vitesse élevée: {speed:.1f} px/frame")
            threat_score += 25
        
        # 2. Accélération brusque
        acceleration = accel.get('magnitude', 0)
        if acceleration > self.THRESHOLDS['high_acceleration']:
            reasons.append(f"Accélération brusque: {acceleration:.1f}")
            threat_score += 20
        
        # 3. Changement de direction soudain
        if dir_change > self.THRESHOLDS['sharp_turn']:
            reasons.append(f"Virage brusque: {dir_change:.1f}°")
            threat_score += 20
            self.suspicious_events.append('sharp_turn')
        
        # 4. Pattern erratique
        recent_suspicious = self.suspicious_events[-10:] if self.suspicious_events else []
        if len(recent_suspicious) >= self.THRESHOLDS['erratic_behavior_count']:
            reasons.append("Pattern de mouvement erratique détecté")
            threat_score += 25
        
        # 5. Analyse de la trajectoire
        if trajectory_info:
            curvature = trajectory_info.get('curvature', 0)
            if curvature > 1.0:
                reasons.append("Trajectoire en zigzag ou circulaire")
                threat_score += 15
        
        # Détermination du statut
        status = self._determine_status(threat_score, len(reasons))
        confidence = min(threat_score / 100, 1.0)
        
        return {
            'status': status,
            'confidence': confidence,
            'reasons': reasons,
            'threat_score': threat_score
        }
    
    def _determine_status(self, threat_score, num_reasons):
        if threat_score >= 60 or num_reasons >= 3:
            return self.STATUS_HOSTILE
        elif threat_score >= 30 or num_reasons >= 2:
            return self.STATUS_SUSPECT
        else:
            return self.STATUS_NORMAL

## 6. Module: Visualisation

In [15]:
class DroneVisualizer:
    """Gère la visualisation des trajectoires et statuts des drones"""
    
    COLORS = {
        'NORMAL': (0, 255, 0),      # Vert
        'SUSPECT': (0, 165, 255),   # Orange
        'HOSTILE': (0, 0, 255)      # Rouge
    }
    
    def __init__(self, trail_length=30, restricted_zone=None):
        self.trail_length = trail_length
        self.restricted_zone = restricted_zone or []
        self.alarm_playing = False
        self.alarm_thread = None
    
    def draw_drone_info(self, frame, drone_data):
        track_id = drone_data.get('track_id')
        x, y = int(drone_data.get('x')), int(drone_data.get('y'))
        status = drone_data.get('status', 'NORMAL')
        trajectory = drone_data.get('trajectory', [])
        predicted = drone_data.get('predicted_trajectory', [])
        velocity = drone_data.get('velocity', {})
        
        color = self.COLORS.get(status, (255, 255, 255))
        
        # 1. Trajectoire historique
        if len(trajectory) > 1:
            pts = np.array(trajectory[-self.trail_length:], np.int32)
            pts = pts.reshape((-1, 1, 2))
            cv2.polylines(frame, [pts], False, color, 2)
        
        # 2. Trajectoire prédite (pointillés)
        if predicted:
            for i in range(len(predicted)-1):
                pt1 = (int(predicted[i][0]), int(predicted[i][1]))
                pt2 = (int(predicted[i+1][0]), int(predicted[i+1][1]))
                cv2.line(frame, pt1, pt2, color, 1, cv2.LINE_AA)
                cv2.circle(frame, pt2, 2, color, -1)
        
        # 3. Carré autour du drone
        square_size = 60
        top_left = (x - square_size // 2, y - square_size // 2)
        bottom_right = (x + square_size // 2, y + square_size // 2)
        cv2.rectangle(frame, top_left, bottom_right, color, 3)
        
        # 4. Informations textuelles
        info_text = f"ID:{track_id} | {status} | V:{velocity.get('magnitude', 0):.1f}"
        cv2.putText(frame, info_text, (x - 60, y - 40), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        return frame

    def draw_restricted_zone(self, frame):
        if not self.restricted_zone:
            return frame
        
        pts = np.array(self.restricted_zone, np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.polylines(frame, [pts], True, (0, 0, 255), 3)
        
        # Overlay semi-transparent rouge
        overlay = frame.copy()
        cv2.fillPoly(overlay, [pts], (0, 0, 255))
        cv2.addWeighted(frame, 0.8, overlay, 0.2, 0, frame)
        
        return frame

    def check_restricted_zone(self, frame, drone_positions):
        zone_violated = False
        if not self.restricted_zone:
            return frame, False
            
        for x, y, _ in drone_positions:
            # Ray casting algorithm simple pour point in polygon
            result = cv2.pointPolygonTest(np.array(self.restricted_zone, np.int32), (x, y), False)
            if result >= 0: # Dedans ou sur le bord
                zone_violated = True
                break
        
        if zone_violated:
            cv2.putText(frame, "ALERTE ZONE INTERDITE!", (50, 100),
                       cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
            # Effet rouge clignotant
            overlay = frame.copy()
            overlay[:] = (0, 0, 255)
            cv2.addWeighted(frame, 0.7, overlay, 0.3, 0, frame)
            
        return frame, zone_violated

## 7. Orchestration Principale

In [16]:
class DroneTrajectoryAnalyzer:
    """Système complet d'analyse de trajectoire"""
    
    def __init__(self, config=None):
        self.config = config or {}
        self.kalman_filters = {}
        self.motion_extractors = {}
        self.trajectory_predictors = {}
        self.behavior_analyzers = {}
        
        restricted_zone = self.config.get('restricted_zone', None)
        self.visualizer = DroneVisualizer(restricted_zone=restricted_zone)
        self.results = defaultdict(list)
        self.prediction_steps = self.config.get('prediction_steps', 10)
    
    def process_tracking_data(self, track_id, frame_num, x, y):
        # Init trackers
        if track_id not in self.kalman_filters:
            self.kalman_filters[track_id] = DroneKalmanFilter()
            self.motion_extractors[track_id] = MotionFeatureExtractor(window_size=10)
            self.trajectory_predictors[track_id] = TrajectoryPredictor(history_size=15)
            self.behavior_analyzers[track_id] = BehaviorAnalyzer()
        
        # 1. Kalman Smooth
        x_smooth, y_smooth = self.kalman_filters[track_id].update(x, y)
        
        # 2. Features
        self.motion_extractors[track_id].add_position(x_smooth, y_smooth, frame_num)
        motion_features = self.motion_extractors[track_id].get_all_features()
        
        # 3. Prédiction
        self.trajectory_predictors[track_id].add_position(x_smooth, y_smooth, frame_num)
        predicted_traj = self.trajectory_predictors[track_id].predict(n_steps=self.prediction_steps)
        curvature = self.trajectory_predictors[track_id].get_trajectory_curvature()
        
        # 4. Behavior
        traj_info = {'curvature': curvature}
        behavior = self.behavior_analyzers[track_id].analyze(motion_features, traj_info)
        
        # Résultat
        result = {
            'track_id': track_id,
            'frame': frame_num,
            'x': x, 'y': y,
            'x_smooth': x_smooth, 'y_smooth': y_smooth,
            'motion_features': motion_features,
            'predicted_trajectory': predicted_traj,
            'behavior': behavior,
            'status': behavior['status'],
            'confidence': behavior['confidence'],
            'threat_score': behavior['threat_score']
        }
        self.results[track_id].append(result)
        return result

    def process_video(self, video_path, tracking_csv, output_path):
        print(f"Traitement de {video_path}...")
        
        # Charger Tracking Data
        tracking_data = defaultdict(list)
        with open(tracking_csv, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                tracking_data[int(row['frame'])].append({
                    'track_id': int(row['track_id']),
                    'x': float(row['x']),
                    'y': float(row['y'])
                })
        
        # Video I/O
        cap = cv2.VideoCapture(video_path)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        
        frame_num = 0
        
        while True:
            ret, frame = cap.read()
            if not ret: break
            
            # Dessiner la zone
            frame = self.visualizer.draw_restricted_zone(frame)
            
            # Proccess Drones
            current_positions = []
            if frame_num in tracking_data:
                for drone in tracking_data[frame_num]:
                    res = self.process_tracking_data(
                        drone['track_id'], frame_num, drone['x'], drone['y']
                    )
                    
                    # Prepare Data pour Viz
                    viz_data = {
                        'track_id': res['track_id'],
                        'x': res['x_smooth'], 'y': res['y_smooth'],
                        'status': res['status'],
                        'velocity': res['motion_features']['instant_velocity'],
                        'trajectory': [(r['x_smooth'], r['y_smooth']) for r in self.results[res['track_id']]],
                        'predicted_trajectory': res['predicted_trajectory']
                    }
                    
                    frame = self.visualizer.draw_drone_info(frame, viz_data)
                    current_positions.append((res['x_smooth'], res['y_smooth'], res['track_id']))
            
            # Check Zones
            frame, violated = self.visualizer.check_restricted_zone(frame, current_positions)
            
            out.write(frame)
            frame_num += 1
            
            if frame_num % 50 == 0:
                print(f"Frame {frame_num}/{total_frames}")
        
        cap.release()
        out.release()
        print(f"Terminé! Vidéo sauvegardée: {output_path}")

## 8. Exécution

In [17]:
if __name__ == "__main__":
    # Définition de la zone restreinte (Carré central)
    center_x, center_y = 518, 663
    size = 250
    restricted_zone = [
        (center_x - size//2, center_y - size//2),
        (center_x + size//2, center_y - size//2),
        (center_x + size//2, center_y + size//2),
        (center_x - size//2, center_y + size//2)
    ]
    
    config = {
        'restricted_zone': restricted_zone,
        'prediction_steps': 15
    }
    
    analyzer = DroneTrajectoryAnalyzer(config)
    
    # Chemins (A ADAPTER)
    VIDEO_INPUT = r"d:\ZAKARIA\STUDIES\AI\newdataset\Projet AI et TI\video_drone.mp4"
    CSV_INPUT = r"d:\ZAKARIA\STUDIES\AI\newdataset\Projet AI et TI\tracking_data.csv"
    VIDEO_OUTPUT = "output_notebook.mp4"
    
    if os.path.exists(VIDEO_INPUT) and os.path.exists(CSV_INPUT):
        analyzer.process_video(VIDEO_INPUT, CSV_INPUT, VIDEO_OUTPUT)
    else:
        print("⚠️ Fichiers d'entrée non trouvés. Vérifiez les chemins.")

Traitement de d:\ZAKARIA\STUDIES\AI\newdataset\Projet AI et TI\video_drone.mp4...
Frame 50/298
Frame 100/298
Frame 150/298
Frame 200/298
Frame 250/298
Terminé! Vidéo sauvegardée: output_notebook.mp4
